In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

class QNetwork(nn.Module):
    def __init__(self, state_size, action_size):
        super(QNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Initialize the Q-Network
state_size = 4   # e.g., CartPole
action_size = 2  # e.g., left or right
q_network = QNetwork(state_size, action_size)
target_network = QNetwork(state_size, action_size)
target_network.load_state_dict(q_network.state_dict())

optimizer = optim.Adam(q_network.parameters(), lr=0.001)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = []
        self.capacity = capacity

    def push(self, experience):
        self.buffer.append(experience)
        if len(self.buffer) > self.capacity:
            self.buffer.pop(0)

    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

    def size(self):
        return len(self.buffer)

replay_buffer = ReplayBuffer(capacity=10000)

In [ ]:
import gymnasium as gym

def compute_td_loss(batch, q_network, target_network, gamma=0.99):
    states, actions, rewards, next_states, dones = zip(*batch)
    states      = torch.tensor(states,      dtype=torch.float32)
    actions     = torch.tensor(actions,     dtype=torch.long)
    rewards     = torch.tensor(rewards,     dtype=torch.float32)
    next_states = torch.tensor(next_states, dtype=torch.float32)
    dones       = torch.tensor(dones,       dtype=torch.bool)
    q_values = q_network(states).gather(1, actions.unsqueeze(1))
    next_q_values = target_network(next_states).max(1)[0]
    target_q_values = rewards + gamma * next_q_values * (~dones)
    loss = torch.mean((q_values - target_q_values.unsqueeze(1)) ** 2)
    return loss

def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        return int(q_network(state_t).argmax().item())

env = gym.make("CartPole-v1")
episodes = 500
batch_size = 64
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
target_update_freq = 10

for episode in range(1, episodes + 1):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay_buffer.push((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        if replay_buffer.size() >= batch_size:
            batch = replay_buffer.sample(batch_size)
            loss = compute_td_loss(batch, q_network, target_network)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % target_update_freq == 0:
        target_network.load_state_dict(q_network.state_dict())

    if episode % 50 == 0:
        print(f"Episode {episode} | Total Reward: {total_reward:.1f} | Epsilon: {epsilon:.3f}")

env.close()
print("Training complete!")

In [ ]:
torch.save(q_network.state_dict(), 'cartpole_dqn.pth')
print("Model saved as cartpole_dqn.pth")

# Download in Colab
try:
    from google.colab import files
    files.download('cartpole_dqn.pth')
except ImportError:
    print("Not in Colab — find cartpole_dqn.pth in your working directory.")